# AutoBrands end-to-end matching tests

Reusable interactive checks for the production DataFrame path:

**Input DataFrame → `normalize.normalize_text` → `matching.Matcher.match_dataframe` → Results DataFrame**

The notebook loads the current catalog and production modules. It does not add matching rules or alter catalog data.

## 1. Configuration

In [5]:
INPUT_DF = None  # Existing pandas DataFrame

BRAND_COL = "brand"  # Or None
MODEL_COL = "model"  # Or None

CATALOG_PATH = "data/vehicle_catalog.csv"
TOP_K = 5
DEBUG = False
MATCHER_VERSION = "v1"  # "v1" or "v2"


## 2. Imports and shared matcher

In [6]:
from pathlib import Path
import sys
from time import perf_counter

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "normalize.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "normalize.py").is_file():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import normalize
import importlib

if str(PROJECT_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.parent))

# Use one normalize module instance for the notebook and matcher.
normalize = importlib.import_module(f"{PROJECT_ROOT.name}.normalize")
sys.modules["normalize"] = normalize
matching = importlib.import_module(f"{PROJECT_ROOT.name}.matching")
from autobrands_v2 import normalize as normalize_v2
from autobrands_v2.matching import Matcher as MatcherV2


In [7]:
catalog_path = Path(CATALOG_PATH)
if not catalog_path.is_absolute():
    catalog_path = PROJECT_ROOT / catalog_path
if not catalog_path.is_file():
    raise FileNotFoundError(f"Vehicle catalog not found: {catalog_path}")
if isinstance(TOP_K, bool) or not isinstance(TOP_K, int) or TOP_K < 1:
    raise ValueError("TOP_K must be a positive integer")
if MATCHER_VERSION not in {"v1", "v2"}:
    raise ValueError("MATCHER_VERSION must be 'v1' or 'v2'")

_matcher_cache = {}
_matcher_init_seconds = {}

def get_matcher(version):
    if version not in {"v1", "v2"}:
        raise ValueError("version must be 'v1' or 'v2'")
    if version not in _matcher_cache:
        started = perf_counter()
        if version == "v1":
            instance = matching.Matcher.from_csv(catalog_path, top_k=TOP_K)
        else:
            instance = MatcherV2.from_csv(catalog_path, top_k=TOP_K)
        _matcher_init_seconds[version] = perf_counter() - started
        _matcher_cache[version] = instance
    return _matcher_cache[version]

# Preserve the selected matcher name used by the existing diagnostics/benchmark cells.
matcher = get_matcher(MATCHER_VERSION)
catalog_df = pd.read_csv(catalog_path, dtype=str, keep_default_na=False)
catalog_pairs = {(brand.upper(), model.upper())
                 for brand, model in zip(catalog_df["brand"], catalog_df["model"])}

effective_configuration = pd.Series({
    "matcher_version": MATCHER_VERSION,
    "min_score": matcher.min_score,
    "min_margin": matcher.min_margin,
    "min_brand": matcher.min_brand,
    "min_model": matcher.min_model,
    "top_k": matcher.top_k,
}, name="effective_value")
display(effective_configuration.to_frame())
print(f"Catalog: {len(catalog_df):,} rows; schema={list(catalog_df.columns)}")


,effective_value
matcher_version,v1
min_score,88.0
min_margin,8.0
min_brand,88.0
min_model,88.0
top_k,5


Catalog: 14,742 rows; schema=['brand', 'brand_aliases', 'model', 'model_aliases', 'vehicle_type']


## 3. Independent synthetic test datasets

The six compact datasets cover brand-only, model-only, joint, ambiguous, variant and dirty/Unicode inputs.

In [8]:
import numpy as np


brands_only = pd.DataFrame(
    {"brand": ["Toyota", "Honda", "Yamaha", "Citroen", "Volkswagen","VW", "ΜΕΡΣΕDES-BEN", "ΥΑΜΑΗΑ", "ΒΜW", "SUΖUΚΙ VΙΤΑR",
               "ΤΟΥΟΤΑ", "Unknown Maker", None, "Toyota"]},
    index=[f"brand-{i}" for i in range(14)],
)

models_only = pd.DataFrame(
    {"model": ["Yaris", "Yaris Cross", "MT-07", "PCX125", "2CV", "SUΖUΚΙ VΙΤΑR",
               "Skaravaios", "Σκαραβα$ος", "Golf +", "Unknown Model", None, "Yaris"]},
    index=[f"model-{i}" for i in range(12)],
)

brand_model = pd.DataFrame([
    {"brand": "Toyota", "model": "Yaris"},
    {"brand": "---", "model": "Yaris"},
    {"brand": "SUΖUΚΙ VΙΤΑR", "model": " "},
    {"brand": "   ", "model": "SUΖUΚΙ VΙΤΑR"},
    {"brand": np.nan,  "model": "Yaris"},
    {"brand": "Honda", "model": "PCX125"},
    {"brand": "Yamaha", "model": "MT09"},
    {"brand": "Audi", "model": "A3 Sportback"},
    {"brand": "Citroen", "model": "Vatrachos"},
    {"brand": "Toyota", "model": "Land Cruiser DIESEL"},
    {"brand": "Kawasaki", "model": "Ninja 400"},
    {"brand": "BMW", "model": "R1250GS"},
    {"brand": "Toyota", "model": "Civic"},
    {"brand": "Toyota", "model": "Yaris"},
], index=[f"pair-{i}" for i in range(14)])

ambiguities = pd.DataFrame([
    {"brand": "Volkswagen", "model": "Skaravaios"},
    {"brand": "Volkswagen", "model": "1300 Skaravaios"},
    {"brand": "Citroen", "model": "Aschimopapo"},
    {"brand": "Renault", "model": "Katrel"},
    {"brand": "Fiat", "model": "500araki"},
    {"brand": "Toyota", "model": "Civic"},
    {"brand": None, "model": "Pioneer"},
    {"brand": "Yamaha", "model": "MT-0"},
    {"brand": "Unknown Maker", "model": "Unknown Model"},
], index=[f"amb-{i}" for i in range(9)])

variants = pd.DataFrame([
    {"brand": "Yamaha", "model": "MT-07"},
    {"brand": "Yamaha", "model": "MT07"},
    {"brand": "Yamaha", "model": "MT-09SP"},
    {"brand": "Yamaha", "model": "MT-09+"},
    {"brand": "Honda", "model": "CBR600RR"},
    {"brand": "Honda", "model": "CBR 600 RR"},
    {"brand": "Toyota", "model": "C-HR"},
    {"brand": "Toyota", "model": "Yaris+"},
    {"brand": "Toyota", "model": "Yaris Cross"},
    {"brand": "Volkswagen", "model": "Golf +"},
    {"brand": "Volkswagen", "model": "Golf+"},
    {"brand": "Fiat", "model": "500-110F"},
], index=[f"variant-{i}" for i in range(12)])

dirty_data = pd.DataFrame([
    {"brand": "  toyota  ", "model": "  yaris  "},
    {"brand": "Ｔｏｙｏｔａ", "model": "Ｙａｒｉｓ"},
    {"brand": "ΤΟΥΟΤΑ", "model": "YARIS"},
    {"brand": "CITROΕN", "model": "C3"},
    {"brand": "YAMHA", "model": "MT07"},
    {"brand": "TOYTA", "model": "YARSI"},
    {"brand": None, "model": "PCX 125"},
    {"brand": "Honda", "model": None},
    {"brand": "???", "model": "@@@"},
    {"brand": "Volkswagen", "model": "Σκαραβαίος"},
    {"brand": "Citroen", "model": "Asximopapo"},
    {"brand": "  toyota  ", "model": "  yaris  "},
], index=[f"dirty-{i}" for i in range(12)])

datasets = {
    "brands_only": brands_only,
    "models_only": models_only,
    "brand_model": brand_model,
    "ambiguities": ambiguities,
    "variants": variants,
    "dirty_data": dirty_data,
}

pd.DataFrame({"rows": {name: len(frame) for name, frame in datasets.items()},
              "columns": {name: ", ".join(frame.columns) for name, frame in datasets.items()}})

,rows,columns
brands_only,14,brand
models_only,12,model
brand_model,14,"brand, model"
ambiguities,9,"brand, model"
variants,12,"brand, model"
dirty_data,12,"brand, model"


## 4. Shared normalization and matching pipeline

Only selected source columns enter the matcher. All other columns and original values stay in the returned DataFrame.

In [ ]:
def _run_pipeline_v1(
    df,
    brand_col=None,
    model_col=None,
    debug=False,
):
    """Normalize selected source columns, match them, and append results."""
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if not df.columns.is_unique:
        raise ValueError("DataFrame column names must be unique")
    if brand_col is None and model_col is None:
        raise ValueError("Supply at least one of brand_col or model_col")
    if not isinstance(debug, bool):
        raise TypeError("debug must be a bool")
    for name in (brand_col, model_col):
        if name is not None and (not isinstance(name, str) or not name):
            raise ValueError("Selected column names must be nonempty strings or None")
        if name is not None and name not in df.columns:
            raise ValueError(f"Selected input column not found: {name!r}")
    if brand_col is not None and brand_col == model_col:
        raise ValueError("brand_col and model_col must be different columns")

    result_columns = []
    if brand_col is not None:
        result_columns.append("corrected_brand")
    if model_col is not None:
        result_columns.append("corrected_model")
    result_columns.extend(["match_status", "match_score", "candidates"])
    debug_columns = ([
        "ab_brand_normalized", "ab_model_normalized", "ab_brand_score",
        "ab_model_score", "ab_score_margin", "ab_match_method",
    ] if debug else [])
    collisions = set(result_columns + debug_columns) & set(df.columns)
    if collisions:
        raise ValueError(f"Output column name collision: {', '.join(sorted(collisions))}")

    # Build only the two canonical matcher inputs; array construction avoids
    # alignment surprises when the source index contains duplicates.
    canonical = pd.DataFrame(index=df.index)
    canonical["brand"] = (df[brand_col].to_numpy(copy=False)
                          if brand_col is not None else [pd.NA] * len(df))
    canonical["model"] = (df[model_col].to_numpy(copy=False)
                          if model_col is not None else [pd.NA] * len(df))

    normalized = canonical.copy()
    for column in ("brand", "model"):
        normalized[column] = normalized[column].map(normalize.normalize_text)
    if brand_col is None:
        normalized = normalized.drop(columns="brand")
    if model_col is None:
        normalized = normalized.drop(columns="model")

    matched = get_matcher("v1").match_dataframe(
        normalized,
        brand_col="brand" if brand_col is not None else None,
        model_col="model" if model_col is not None else None,
        top_k=TOP_K, debug=debug,
    )
    result = df.copy(deep=False)
    for name in result_columns + debug_columns:
        if name in matched.columns:
            result[name] = matched[name].to_numpy(copy=False)
    if len(result) != len(df) or not result.index.equals(df.index):
        raise AssertionError("Pipeline changed row count, order, or index")
    return result

def _run_pipeline_v2(df, brand_col=None, model_col=None, debug=False):
    """Send raw selected inputs to V2; it owns its independent normalization."""
    return get_matcher("v2").match_dataframe(
        df, brand_col=brand_col, model_col=model_col, top_k=TOP_K, debug=debug
    )


def run_pipeline_for_version(df, brand_col=None, model_col=None, debug=False, *, version):
    if version == "v1":
        return _run_pipeline_v1(df, brand_col=brand_col, model_col=model_col, debug=debug)
    if version == "v2":
        return _run_pipeline_v2(df, brand_col=brand_col, model_col=model_col, debug=debug)
    raise ValueError("version must be 'v1' or 'v2'")


def run_pipeline(df, brand_col=None, model_col=None, debug=False):
    """Run the configured matcher version on the existing DataFrame API."""
    return run_pipeline_for_version(
        df, brand_col=brand_col, model_col=model_col, debug=debug,
        version=MATCHER_VERSION,
    )


## 5. Real DataFrame usage

The configured `INPUT_DF` runs when supplied. This example is guarded and runs only if `my_dataframe` exists:

```python
result = run_pipeline(
    my_dataframe,
    brand_col="manufacturer",
    model_col="vehicle_name",
)
display(result.head(20))
```

Single-column inputs are supported with `run_pipeline(df, brand_col="make")` or `run_pipeline(df, model_col="model_name")`.

In [10]:
if INPUT_DF is not None:
    configured_result = run_pipeline(
        INPUT_DF,
        brand_col=BRAND_COL,
        model_col=MODEL_COL,
        debug=DEBUG,
    )
    display(configured_result.head(20))

# The following example runs only if a DataFrame named my_dataframe exists.
if "my_dataframe" in globals() and isinstance(my_dataframe, pd.DataFrame):
    result = run_pipeline(
        my_dataframe,
        brand_col="manufacturer",
        model_col="vehicle_name",
    )
    display(result.head(20))

## 6. Results and candidate inspection

In [11]:
RESULT_COLUMNS = ["corrected_brand", "corrected_model", "match_status",
                  "match_score", "candidates"]
DEBUG_COLUMNS = ["ab_brand_normalized", "ab_model_normalized",
                 "ab_brand_score", "ab_model_score",
                 "ab_score_margin", "ab_match_method"]
FULL_MATCHES = {"EXACT_MATCH", "ALIAS_MATCH", "FUZZY_MATCH"}

def result_view(result_df: pd.DataFrame, debug: bool = False) -> pd.DataFrame:
    output_names = RESULT_COLUMNS + (DEBUG_COLUMNS if debug else [])
    source_columns = [name for name in result_df.columns if name not in output_names]
    return result_df[source_columns + [name for name in output_names
                                      if name in result_df.columns]].copy()

def show_candidates(result_df: pd.DataFrame, row_index):
    selected = result_df.loc[row_index]
    if isinstance(selected, pd.DataFrame):
        raise ValueError("row_index must identify exactly one row")
    candidates = selected["candidates"] or []
    columns = ["canonical_brand", "score"]
    if candidates and "model" in candidates[0]:
        columns.insert(1, "canonical_model")
    if not candidates:
        display(Markdown(f"No ranked candidates for row `{row_index}`."))
        return pd.DataFrame(columns=columns)
    frame = pd.DataFrame(candidates[:TOP_K]).rename(
        columns={"brand": "canonical_brand", "model": "canonical_model"}
    )[columns]
    display(frame)
    return frame

def diagnostics(result_df: pd.DataFrame) -> pd.Series:
    statuses = result_df["match_status"]
    return pd.Series({
        "total_rows": len(result_df),
        "accepted_matches": int(statuses.isin(FULL_MATCHES | {"MATCHED"}).sum()),
        "ambiguous_cases": int(statuses.eq("AMBIGUOUS").sum()),
        "unresolved_cases": int(statuses.eq("UNRESOLVED").sum()),
        "multiple_candidates": int(result_df["candidates"].map(len).gt(1).sum()),
    })

def status_counts(result_df: pd.DataFrame) -> pd.Series:
    statuses = ["MATCHED", "EXACT_MATCH", "ALIAS_MATCH", "FUZZY_MATCH",
                "PARTIAL_MATCH", "AMBIGUOUS", "CONFLICT", "UNRESOLVED"]
    return result_df["match_status"].value_counts().reindex(statuses, fill_value=0)

def accepted_matches(result_df: pd.DataFrame) -> pd.DataFrame:
    return result_df.loc[result_df["match_status"].isin(FULL_MATCHES | {"MATCHED"})]

def ambiguous_results(result_df: pd.DataFrame) -> pd.DataFrame:
    return result_df.loc[result_df["match_status"].eq("AMBIGUOUS")]

def unresolved_results(result_df: pd.DataFrame) -> pd.DataFrame:
    return result_df.loc[result_df["match_status"].eq("UNRESOLVED")]

## 7. Execute all six synthetic datasets

In [ ]:
input_snapshots = {name: frame.copy(deep=True) for name, frame in datasets.items()}
dataset_columns = {
    "brands_only": ("brand", None),
    "models_only": (None, "model"),
    "brand_model": ("brand", "model"),
    "ambiguities": ("brand", "model"),
    "variants": ("brand", "model"),
    "dirty_data": ("brand", "model"),
}
results_by_version = {
    version: {
        name: run_pipeline_for_version(
            frame,
            brand_col=dataset_columns[name][0],
            model_col=dataset_columns[name][1],
            version=version,
        )
        for name, frame in datasets.items()
    }
    for version in ("v1", "v2")
}
results = results_by_version[MATCHER_VERSION]
accepted_statuses = FULL_MATCHES | {"MATCHED"}
comparison_rows = []
for name, source in datasets.items():
    left, right = results_by_version["v1"][name], results_by_version["v2"][name]
    identity_columns = [column for column in ("corrected_brand", "corrected_model")
                        if column in left.columns and column in right.columns]
    differing_acceptance_or_identity = 0
    for position in range(len(left)):
        left_accepted = left.iloc[position]["match_status"] in accepted_statuses
        right_accepted = right.iloc[position]["match_status"] in accepted_statuses
        identities_differ = (tuple(left.iloc[position][column] for column in identity_columns)
                             != tuple(right.iloc[position][column] for column in identity_columns))
        differing_acceptance_or_identity += int(
            left_accepted != right_accepted or (left_accepted and right_accepted and identities_differ)
        )
    comparison_rows.append({
        "dataset": name,
        "rows": len(source),
        "v1_accepted": int(left["match_status"].isin(accepted_statuses).sum()),
        "v2_accepted": int(right["match_status"].isin(accepted_statuses).sum()),
        "different_status": int((left["match_status"].to_numpy()
                                  != right["match_status"].to_numpy()).sum()),
        "different_accepted_identity": differing_acceptance_or_identity,
    })
comparison_summary = pd.DataFrame(comparison_rows).set_index("dataset")

diagnostic_table = pd.DataFrame(
    {name: diagnostics(frame) for name, frame in results.items()}
).T
status_table = pd.DataFrame(
    {name: status_counts(frame) for name, frame in results.items()}
).T.add_prefix("status_")
display(diagnostic_table.join(status_table))
display(comparison_summary)
display(Markdown(f"Results shown below use **{MATCHER_VERSION}**; both versions processed every synthetic dataset."))


In [ ]:
for dataset_name, result_df in results.items():
    display(Markdown(f"### {dataset_name}"))
    display(result_view(result_df))

### Inspect ranked candidates and filters

In [ ]:
candidate_example = show_candidates(results["ambiguities"], "amb-0")

debug_result = run_pipeline(brand_model.head(3), brand_col="brand", model_col="model", debug=True)
display(Markdown("**Optional debug view**"))
display(result_view(debug_result, debug=True))


display(Markdown("**Ambiguous inputs**"))
display(result_view(ambiguous_results(results["ambiguities"])))
display(Markdown("**Accepted inputs**"))
display(result_view(accepted_matches(results["brand_model"])))
display(Markdown("**Unresolved inputs**"))
display(result_view(unresolved_results(results["dirty_data"])))

## 8. Structural and real-DataFrame validation

In [ ]:
for name, source_df in datasets.items():
    result_df = results[name]
    assert len(result_df) == len(source_df), f"{name}: row count changed"
    assert result_df.index.equals(source_df.index), f"{name}: index/order changed"
    pd.testing.assert_frame_equal(source_df, input_snapshots[name])
    brand, model = dataset_columns[name]
    expected = list(source_df.columns)
    if brand is not None:
        expected.append("corrected_brand")
    if model is not None:
        expected.append("corrected_model")
    expected += ["match_status", "match_score", "candidates"]
    assert list(result_df.columns) == expected, f"{name}: unexpected output columns"
    assert all(len(candidates) <= TOP_K for candidates in result_df["candidates"])

# Arbitrary Unicode/spaced source names, extra columns and duplicate indices.
unicode_source = pd.DataFrame({
    "Κατασκευαστής": ["ΤΟΥΟΤΑ", "Toyota", "Toyota", "???", None, "  	 "],
    "Vehicle name / model": ["Yaris", "", "Yaris", "@@@", "Σκαθάρι 1500", "  	 "],
    "claim notes": ["keep", "blank", "duplicate", "noise", "greek", "whitespace"],
}, index=[7, 7, 9, 10, 10, 11])
unicode_snapshot = unicode_source.copy(deep=True)
unicode_result = run_pipeline(
    unicode_source,
    brand_col="Κατασκευαστής",
    model_col="Vehicle name / model",
)
assert len(unicode_result) == len(unicode_source)
assert unicode_result.index.equals(unicode_source.index)
pd.testing.assert_frame_equal(unicode_source, unicode_snapshot)
assert unicode_result["claim notes"].tolist() == unicode_source["claim notes"].tolist()
assert unicode_result.loc[7, "corrected_brand"].iloc[0] == "TOYOTA"
assert unicode_result.loc[7, "corrected_brand"].iloc[1] == "TOYOTA"

# One selected field, including arbitrary names; the unselected correction is omitted.
brand_named = run_pipeline(unicode_source, brand_col="Κατασκευαστής")
model_named = run_pipeline(unicode_source, model_col="Vehicle name / model")
assert "corrected_brand" in brand_named and "corrected_model" not in brand_named
assert "corrected_model" in model_named and "corrected_brand" not in model_named

# Empty input preserves its schema and appends the selected result fields.
empty_source = pd.DataFrame(columns=["make name", "model name", "source note"])
empty_result = run_pipeline(empty_source, brand_col="make name", model_col="model name")
assert empty_result.empty and len(empty_result.columns) == 8

# Missing-column selection and result-name collisions fail before matching.
for bad_call in (
    lambda: run_pipeline(unicode_source),
    lambda: run_pipeline(unicode_source, brand_col="not here"),
    lambda: run_pipeline(pd.DataFrame({"make": ["Toyota"],
                                       "corrected_brand": ["keep"]}), brand_col="make"),
):
    try:
        bad_call()
    except ValueError:
        pass
    else:
        raise AssertionError("Expected input/schema validation error")

# Configured real input is optional; synthetic datasets always use the same API.
assert INPUT_DF is None or isinstance(INPUT_DF, pd.DataFrame)
print("All DataFrame API, preservation, schema and synthetic-dataset assertions passed.")

## 8. Execution findings

Assertion failures indicate a pipeline or matcher defect. Unresolved deliberately unknown inputs are listed separately as potential catalog-coverage or negative-test cases.

In [ ]:
coverage_rows = []
for dataset_name, result_df in results.items():
    mask = result_df["match_status"].eq("UNRESOLVED")
    for row_index, row in result_df.loc[mask].iterrows():
        coverage_rows.append({
            "dataset": dataset_name,
            "row_index": row_index,
            "brand": row.get("brand"),
            "model": row.get("model"),
        })

print("Matcher bugs detected by notebook assertions: none.")
if coverage_rows:
    display(Markdown("**Potential catalog coverage / deliberate negative cases**"))
    display(pd.DataFrame(coverage_rows))
else:
    print("Potential catalog coverage gaps among these cases: none.")

In [ ]:
from normalize import comparison_forms, normalize_text


samples = ["ΒΜW", "ΥΑΜΑΗΑ", "ΟΡΕL", "ΟΠΕΛ", "ΣΚΑΘΑΡΙ"]

for value in samples:
    print(value)
    print("normalized:", normalize_text(value))
    print("alternatives:", comparison_forms(value))

## 9. V1 / V2 comparison and V2 performance

Use `compare_versions(...)` for any real or synthetic DataFrame. Scores belong to each version's own scale and should not be compared as probabilities. In V2, `match_score` is the top candidate's ranking score; it is not a probability or acceptance confidence. Exact canonical/alias, formatting and Unicode-equivalent comparisons may score 100, and an `AMBIGUOUS` row may also score 100 when candidates collide. `match_status` and the score margin determine acceptance; `debug=True` reports the comparison basis in `ab_match_method`. The timing cell profiles V2 on 1,000 repeated rows; V1 is sampled on 20 rows because its legacy matcher is much slower, then reported as a per-1,000-row estimate.

In [ ]:
def compare_versions(df, brand_col=None, model_col=None, disagreements_only=False):
    """Compare both versions on identical raw source columns."""
    left = run_pipeline_for_version(df, brand_col, model_col, version="v1")
    right = run_pipeline_for_version(df, brand_col, model_col, version="v2")
    rows = pd.DataFrame(index=df.index)
    if brand_col is not None:
        rows["brand"] = df[brand_col].to_numpy(copy=False)
    if model_col is not None:
        rows["model"] = df[model_col].to_numpy(copy=False)
    for version, frame in (("v1", left), ("v2", right)):
        for column in ("corrected_brand", "corrected_model", "match_status", "match_score"):
            if column in frame:
                rows[f"{version}_{column}"] = frame[column].to_numpy(copy=False)
    identity_fields = [column for column in ("corrected_brand", "corrected_model")
                       if column in left and column in right]
    accepted = FULL_MATCHES | {"MATCHED"}
    rows["accepted_identity_differs"] = [
        ((left.iloc[i]["match_status"] in accepted)
         != (right.iloc[i]["match_status"] in accepted))
        or ((left.iloc[i]["match_status"] in accepted)
            and (right.iloc[i]["match_status"] in accepted)
            and tuple(left.iloc[i][field] for field in identity_fields)
            != tuple(right.iloc[i][field] for field in identity_fields))
        for i in range(len(df))
    ]
    if disagreements_only:
        rows = rows.loc[rows["accepted_identity_differs"]]
    return rows


# Profile identical repeated rows; stats are incremental for the V2 matcher.
import tracemalloc
from time import perf_counter

profile_seed = pd.concat(
    [frame.reindex(columns=["brand", "model"]) for frame in datasets.values()],
    ignore_index=True,
)
profile_source = pd.concat(
    [profile_seed] * ((1000 + len(profile_seed) - 1) // len(profile_seed)),
    ignore_index=True,
).iloc[:1000].copy()
performance_rows = []
for version in ("v1", "v2"):
    instance = get_matcher(version)
    sample_size = 20 if version == "v1" else 1000
    profile_sample = profile_source.iloc[:sample_size]
    if version == "v2":
        # Clear only the V2 score cache so this profile counts the needed comparisons.
        instance._score_cache.clear()
    weighted_before = instance.stats.get("weighted_comparisons", 0) if version == "v2" else None
    retrieved_before = instance.stats.get("candidate_ids_retrieved", 0) if version == "v2" else None
    started = perf_counter()
    profile_result = run_pipeline_for_version(
        profile_sample, brand_col="brand", model_col="model", version=version
    )
    elapsed = perf_counter() - started
    weighted_after = instance.stats.get("weighted_comparisons", 0) if version == "v2" else None
    retrieved_after = instance.stats.get("candidate_ids_retrieved", 0) if version == "v2" else None
    tracemalloc.start()
    _memory_result = run_pipeline_for_version(
        profile_sample, brand_col="brand", model_col="model", version=version
    )
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    performance_rows.append({
        "version": version,
        "matcher_init_seconds": _matcher_init_seconds[version],
        "rows_profiled": len(profile_result),
        "seconds_per_1000_rows": elapsed * 1000 / max(1, len(profile_result)),
        "peak_traced_mb": peak_bytes / (1024 * 1024),
        "weighted_comparisons_per_row": (
            (weighted_after - weighted_before) / len(profile_result)
            if version == "v2" else None
        ),
        "candidate_ids_retrieved_per_row": (
            (retrieved_after - retrieved_before) / len(profile_result)
            if version == "v2" else None
        ),
        "catalog_reduction_fraction": (
            1 - ((retrieved_after - retrieved_before)
                 / max(1, len(profile_result) * instance.identity_count))
            if version == "v2" else None
        ),
    })
display(pd.DataFrame(performance_rows).set_index("version"))


## 10. Benchmark / Evaluation

This section is separate from the production `run_pipeline()` examples above. The repository contains no reviewed real claim labels, so the checked-in [annotation template](../benchmark/ground_truth_template.csv) is intentionally empty. Pending or unresolvable rows are never used as accuracy labels, and matcher predictions must never be copied into expected fields.

The current matcher consumes comparison alternatives without retaining their provenance. Strategy A therefore reports end-to-end production metrics. B–D are explicitly **candidate-generation-only** experiments; their candidates are not accepted identities and their output must not be used to claim end-to-end precision, coverage, or accuracy. Strategy C is a no-op against this baseline because the current `comparison_forms()` already includes bounded visual/phonetic alternatives. `disarm` is optional and experimental; no dependency is installed by this notebook.

Only call `evaluate_benchmark(...)` after adding independently reviewed annotations. The full evaluation is opt-in and is not run by the ordinary DataFrame pipeline.


In [ ]:
from contextlib import contextmanager
from hashlib import sha256
from time import perf_counter

GROUND_TRUTH_PATH = PROJECT_ROOT / "benchmark" / "ground_truth_template.csv"
GROUND_TRUTH_COLUMNS = [
    "row_id", "input_mode", "brand", "model", "expected_brand", "expected_model",
    "evaluation_category", "catalog_covered", "annotation_status", "split",
]
VALID_INPUT_MODES = {"brand_only", "model_only", "brand_model"}
VALID_ANNOTATION_STATUS = {"reviewed", "pending", "unresolvable"}
VALID_SPLITS = {"development", "final_test", "unassigned"}
ACCEPTED_STATUSES = {"MATCHED", "EXACT_MATCH", "ALIAS_MATCH", "FUZZY_MATCH"}
CATALOG_BRANDS = {normalize.normalize_text(value) for value in catalog_df["brand"]}
CATALOG_IDENTITIES = {
    (normalize.normalize_text(brand), normalize.normalize_text(model))
    for brand, model in zip(catalog_df["brand"], catalog_df["model"])
}
CATALOG_SHA256 = sha256(catalog_path.read_bytes()).hexdigest()

try:
    import disarm
    DISARM_AVAILABLE = True
    DISARM_VERSION = getattr(disarm, "__version__", "installed; version unavailable")
except ImportError:
    disarm = None
    DISARM_AVAILABLE = False
    DISARM_VERSION = "not installed"

def _blank(value):
    return value is None or (isinstance(value, str) and not value.strip()) or pd.isna(value)

def validate_ground_truth(frame: pd.DataFrame) -> pd.DataFrame:
    """Validate user annotations without creating or inferring any labels."""
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("Ground truth must be a pandas DataFrame")
    missing = [name for name in GROUND_TRUTH_COLUMNS if name not in frame.columns]
    if missing:
        raise ValueError(f"Ground truth is missing columns: {missing}")
    checked = frame[GROUND_TRUTH_COLUMNS].fillna("").astype(str).copy()
    if checked["row_id"].str.strip().eq("").any() or checked["row_id"].duplicated().any():
        raise ValueError("row_id values must be nonempty and unique")
    if not set(checked["input_mode"]).issubset(VALID_INPUT_MODES):
        raise ValueError("input_mode must be brand_only, model_only, or brand_model")
    if not set(checked["annotation_status"]).issubset(VALID_ANNOTATION_STATUS):
        raise ValueError("annotation_status must be reviewed, pending, or unresolvable")
    if not set(checked["split"]).issubset(VALID_SPLITS):
        raise ValueError("split must be development, final_test, or unassigned")
    if not set(checked["catalog_covered"].str.lower()).issubset({"", "true", "false", "yes", "no"}):
        raise ValueError("catalog_covered must be blank, true/false, or yes/no")

    split_by_identity = {}
    split_by_source = {}
    for row in checked.itertuples(index=False):
        expected_brand = normalize.normalize_text(row.expected_brand)
        expected_model = normalize.normalize_text(row.expected_model)
        covered = row.catalog_covered.strip().lower() in {"true", "yes"}
        not_covered = row.catalog_covered.strip().lower() in {"false", "no"}
        if row.annotation_status == "reviewed" and covered:
            if row.input_mode == "brand_only":
                if not expected_brand or expected_brand not in CATALOG_BRANDS or expected_model:
                    raise ValueError(f"Invalid reviewed brand-only identity at row_id={row.row_id}")
            elif not expected_brand or not expected_model or (expected_brand, expected_model) not in CATALOG_IDENTITIES:
                raise ValueError(f"Expected identity is absent from the catalog at row_id={row.row_id}")
        if row.annotation_status == "reviewed" and not_covered and expected_brand and expected_model:
            if (expected_brand, expected_model) in CATALOG_IDENTITIES:
                raise ValueError(f"catalog_covered contradicts catalog at row_id={row.row_id}")

        if row.split in {"development", "final_test"}:
            if expected_brand and (row.input_mode == "brand_only" or expected_model):
                identity = (row.input_mode, expected_brand, expected_model)
                previous = split_by_identity.setdefault(identity, row.split)
                if previous != row.split:
                    raise ValueError("Same labeled identity crosses development/final_test splits")
            source_brand = normalize.normalize_text(row.brand) or ""
            source_model = normalize.normalize_text(row.model) or ""
            source_key = (row.input_mode, "".join(c for c in source_brand if c.isalnum()),
                          "".join(c for c in source_model if c.isalnum()))
            previous = split_by_source.setdefault(source_key, row.split)
            if previous != row.split:
                raise ValueError("Duplicate/near-duplicate input crosses development/final_test splits")
    return checked

def load_ground_truth(path=GROUND_TRUTH_PATH) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    return validate_ground_truth(frame)

def deterministic_sample(frame: pd.DataFrame, max_rows=None, random_state=2026) -> pd.DataFrame:
    """Sample per input mode with a fixed seed; retain stable row IDs and order."""
    if max_rows is None or len(frame) <= max_rows:
        return frame.sort_values("row_id", kind="stable").reset_index(drop=True)
    if isinstance(max_rows, bool) or not isinstance(max_rows, int) or max_rows < 1:
        raise ValueError("max_rows must be a positive integer or None")
    groups = []
    modes = sorted(frame["input_mode"].unique())
    quotas = {mode: max_rows // len(modes) for mode in modes}
    for mode in modes[:max_rows % len(modes)]:
        quotas[mode] += 1
    for mode in modes:
        group = frame.loc[frame["input_mode"].eq(mode)].sort_values("row_id", kind="stable")
        n = min(len(group), quotas[mode])
        if n:
            groups.append(group.sample(n=n, random_state=random_state))
    if not groups:
        return frame.iloc[0:0].copy()
    return pd.concat(groups).sort_values("row_id", kind="stable").reset_index(drop=True)

GT = load_ground_truth()
reviewed_count = int(GT["annotation_status"].eq("reviewed").sum())
print(f"Ground truth rows: {len(GT)}; reviewed rows: {reviewed_count}")
print(f"Catalog SHA-256: {CATALOG_SHA256}; disarm: {DISARM_VERSION}")
if reviewed_count == 0:
    print("No reviewed labels: strategy accuracy/precision/coverage claims are unavailable.")


In [ ]:
def _disarm_alternatives(value):
    """Return experimental Latin renderings; never write them into the catalog."""
    if disarm is None or _blank(value):
        return ()
    raw = str(value)
    outputs = []
    for candidate in (disarm.transliterate(raw), disarm.normalize_confusables(raw)):
        normalized = normalize.normalize_text(candidate)
        if normalized and normalized not in outputs:
            outputs.append(normalized)
    return tuple(outputs)

@contextmanager
def _comparison_strategy(strategy):
    """Temporarily add experimental alternatives for catalog indexing/querying."""
    original = normalize.comparison_forms
    if strategy not in {"B", "D"} or not DISARM_AVAILABLE:
        yield
        return
    def expanded(value):
        baseline = list(original(value))
        for candidate in _disarm_alternatives(value):
            if candidate not in baseline and len(baseline) < 8:
                baseline.append(candidate)
        return tuple(baseline)
    normalize.comparison_forms = expanded
    try:
        yield
    finally:
        normalize.comparison_forms = original

_STRATEGY_MATCHER_CACHE = {"A": get_matcher("v1"), "C": get_matcher("v1")}

def _strategy_matcher(strategy):
    if strategy in _STRATEGY_MATCHER_CACHE:
        return _STRATEGY_MATCHER_CACHE[strategy]
    if not DISARM_AVAILABLE:
        return None
    with _comparison_strategy(strategy):
        strategy_matcher = matching.Matcher.from_csv(
            catalog_path,
            min_score=matcher.min_score,
            min_margin=matcher.min_margin,
            min_brand=matcher.min_brand,
            min_model=matcher.min_model,
            top_k=matcher.top_k,
        )
    _STRATEGY_MATCHER_CACHE[strategy] = strategy_matcher
    return strategy_matcher

def _query_alternatives(raw, strategy):
    base = list(normalize.comparison_forms(raw))
    if strategy in {"B", "D"}:
        for value in _disarm_alternatives(raw):
            if value not in base and len(base) < 8:
                base.append(value)
    return tuple(base) or ("",)

def _rank_candidates_for_row(row, strategy, strategy_matcher):
    """Candidate generation only: do not return or count Matcher decisions as accepted."""
    brand = row.brand if isinstance(row.brand, str) else ""
    model = row.model if isinstance(row.model, str) else ""
    brand_alts = _query_alternatives(brand, strategy)
    model_alts = _query_alternatives(model, strategy)
    best_by_id = {}
    for brand_alt in brand_alts:
        for model_alt in model_alts:
            decision = strategy_matcher.resolve(
                brand or None, model or None,
                brand_latin=brand_alt if brand else None,
                model_latin=model_alt if model else None,
            )
            for candidate in decision.top_candidates:
                previous = best_by_id.get(candidate.identity_id)
                quality = (candidate.score, candidate.model_score, candidate.brand_score)
                if previous is None or quality > (previous.score, previous.model_score, previous.brand_score):
                    best_by_id[candidate.identity_id] = candidate
    ranked = sorted(
        best_by_id.values(),
        key=lambda item: (-item.score, -item.model_score,
                          strategy_matcher.identities[item.identity_id].brand_key,
                          strategy_matcher.identities[item.identity_id].model_key),
    )[:TOP_K]
    return [
        {"brand": strategy_matcher.identities[item.identity_id].brand,
         "model": strategy_matcher.identities[item.identity_id].model,
         "score": item.score}
        for item in ranked
    ]


In [ ]:
def _ground_truth_inputs(frame, mode):
    subset = frame.loc[frame["input_mode"].eq(mode)].copy()
    if subset.empty:
        return subset, subset.copy()
    source = pd.DataFrame({
        "row_id": subset["row_id"].to_list(),
        "brand": subset["brand"].to_list(),
        "model": subset["model"].to_list(),
    })
    if mode == "brand_only":
        result = run_pipeline(source, brand_col="brand")
    elif mode == "model_only":
        result = run_pipeline(source, model_col="model")
    else:
        result = run_pipeline(source, brand_col="brand", model_col="model")
    return subset.reset_index(drop=True), result.reset_index(drop=True)

def _is_candidate_correct(candidates, row):
    expected_brand = normalize.normalize_text(row.expected_brand)
    expected_model = normalize.normalize_text(row.expected_model)
    for candidate in (candidates or []):
        brand = normalize.normalize_text(candidate.get("brand", ""))
        model = normalize.normalize_text(candidate.get("model", ""))
        if row.input_mode == "brand_only":
            if expected_brand and brand == expected_brand:
                return True
        elif expected_brand and expected_model and (brand, model) == (expected_brand, expected_model):
            return True
    return False

def _accepted_identity_matches(result, row):
    if result.get("match_status") not in ACCEPTED_STATUSES:
        return False
    expected_brand = normalize.normalize_text(row.expected_brand)
    expected_model = normalize.normalize_text(row.expected_model)
    if row.input_mode == "brand_only":
        return normalize.normalize_text(result.get("corrected_brand", "")) == expected_brand
    candidates = result.get("candidates") or []
    if candidates:
        top = candidates[0]
        return (normalize.normalize_text(top.get("brand", "")),
                normalize.normalize_text(top.get("model", ""))) == (expected_brand, expected_model)
    return (normalize.normalize_text(result.get("corrected_brand", "")),
            normalize.normalize_text(result.get("corrected_model", ""))) == (expected_brand, expected_model)

def _metric_row(strategy, mode, labeled, outputs, elapsed, evaluation_kind, runtime_rows=None):
    n = len(labeled)
    accepted_rows = []
    correct_accepted = 0
    top5_correct = 0
    ambiguous = unresolved = partial = partial_correct = 0
    for source, output in zip(labeled.itertuples(index=False), outputs):
        status = output.get("match_status", "")
        is_correct = _accepted_identity_matches(output, source)
        if status in ACCEPTED_STATUSES:
            accepted_rows.append((source, output))
            correct_accepted += int(is_correct)
        top5_correct += int(_is_candidate_correct(output.get("candidates", []), source))
        ambiguous += int(status == "AMBIGUOUS")
        unresolved += int(status == "UNRESOLVED")
        if status == "PARTIAL_MATCH":
            partial += 1
            expected_brand = normalize.normalize_text(source.expected_brand)
            expected_model = normalize.normalize_text(source.expected_model)
            brand_out = normalize.normalize_text(output.get("corrected_brand", ""))
            model_out = normalize.normalize_text(output.get("corrected_model", ""))
            has_correct_field = ((brand_out and brand_out == expected_brand)
                                 or (model_out and model_out == expected_model))
            no_wrong_field = (not brand_out or brand_out == expected_brand) and (not model_out or model_out == expected_model)
            partial_correct += int(bool(has_correct_field and no_wrong_field))
    incorrect = len(accepted_rows) - correct_accepted
    return {
        "strategy": strategy, "input_mode": mode, "evaluation_kind": evaluation_kind,
        "n_reviewed_catalog_covered": n,
        "exact_identity_accuracy": correct_accepted / n if n else None,
        "auto_match_precision": correct_accepted / len(accepted_rows) if accepted_rows else None,
        "auto_match_coverage": len(accepted_rows) / n if n else None,
        "false_auto_match_count": incorrect,
        "false_auto_match_rate": incorrect / len(accepted_rows) if accepted_rows else None,
        "top5_candidate_recall": top5_correct / n if n else None,
        "ambiguous_rate": ambiguous / n if n else None,
        "unresolved_rate": unresolved / n if n else None,
        "partial_count": partial,
        "partial_correct_count": partial_correct,
        "partial_correct_rate": partial_correct / partial if partial else None,
        "runtime_seconds_per_1000": elapsed * 1000 / runtime_rows if runtime_rows else None,
    }

def _candidate_signature(candidates):
    return tuple((normalize.normalize_text(item.get("brand", "")),
                  normalize.normalize_text(item.get("model", "")), item.get("score"))
                 for item in (candidates or []))

def evaluate_benchmark(ground_truth, *, max_rows=None, random_state=2026):
    """Run A end-to-end; B-D remain candidate-generation-only by design."""
    frame = deterministic_sample(validate_ground_truth(ground_truth), max_rows, random_state)
    if frame.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    eligible = frame.loc[
        frame["annotation_status"].eq("reviewed")
        & frame["catalog_covered"].str.lower().isin({"true", "yes"})
    ].copy()
    strategies = ["A", "B", "C", "D"]
    strategy_predictions = {}
    metric_rows = []
    baseline_by_id = {}
    baseline_elapsed_by_mode = {}
    for mode in sorted(frame["input_mode"].unique()):
        start = perf_counter()
        selected, outputs = _ground_truth_inputs(frame, mode)
        baseline_elapsed_by_mode[mode] = perf_counter() - start
        for _, output in outputs.iterrows():
            baseline_by_id[output["row_id"]] = output.to_dict()
    strategy_predictions["A"] = baseline_by_id

    for mode in sorted(frame["input_mode"].unique()):
        target_rows = eligible.loc[eligible["input_mode"].eq(mode)]
        if target_rows.empty:
            continue
        baseline_outputs = [baseline_by_id[row_id] for row_id in target_rows["row_id"]]
        runtime_rows = int(frame["input_mode"].eq(mode).sum())
        metric_rows.append(_metric_row("A", mode, target_rows, baseline_outputs,
                                       baseline_elapsed_by_mode.get(mode, 0.0), "end_to_end",
                                       runtime_rows=runtime_rows))

    for strategy in ("B", "C", "D"):
        if strategy in {"B", "D"} and not DISARM_AVAILABLE:
            continue
        generator = _strategy_matcher(strategy)
        generated = {}
        elapsed_by_mode = {}
        with _comparison_strategy(strategy):
            for mode in sorted(frame["input_mode"].unique()):
                started = perf_counter()
                for row in frame.loc[frame["input_mode"].eq(mode)].itertuples(index=False):
                    generated[row.row_id] = _rank_candidates_for_row(row, strategy, generator)
                elapsed_by_mode[mode] = perf_counter() - started
        strategy_predictions[strategy] = generated
        for mode in sorted(frame["input_mode"].unique()):
            target_rows = eligible.loc[eligible["input_mode"].eq(mode)]
            if target_rows.empty:
                continue
            preliminary_outputs = [{"candidates": generated[row_id]} for row_id in target_rows["row_id"]]
            runtime_rows = int(frame["input_mode"].eq(mode).sum())
            metric_rows.append(_metric_row(strategy, mode, target_rows, preliminary_outputs,
                                           elapsed_by_mode.get(mode, 0.0), "candidate_generation_only",
                                           runtime_rows=runtime_rows))

    metric_table = pd.DataFrame(metric_rows)
    disagreements = []
    for row in frame.itertuples(index=False):
        baseline = baseline_by_id[row.row_id]
        baseline_candidates = baseline.get("candidates", [])
        baseline_identity = _accepted_identity_matches(baseline, row)
        for strategy, generated in strategy_predictions.items():
            if strategy == "A" or row.row_id not in generated:
                continue
            alt_candidates = generated[row.row_id]
            if _candidate_signature(baseline_candidates) == _candidate_signature(alt_candidates):
                continue
            reviewed = row.annotation_status == "reviewed" and row.catalog_covered.lower() in {"true", "yes"}
            expected = f"{row.expected_brand} / {row.expected_model}".strip(" /")
            disagreements.append({
                "row_id": row.row_id, "strategy": strategy,
                "input_mode": row.input_mode, "brand": row.brand, "model": row.model,
                "expected_identity": expected if reviewed else "",
                "annotation_status": row.annotation_status,
                "error_category": row.evaluation_category or "Other",
                "baseline_status": baseline.get("match_status", ""),
                "baseline_candidates": baseline_candidates,
                "baseline_accepted_identity_correct": baseline_identity if reviewed else None,
                "alternative_top5_candidates": alt_candidates,
                "comparison_scope": "candidate-generation only; no acceptance decision",
            })
    return metric_table, pd.DataFrame(disagreements), pd.DataFrame(strategy_predictions)


### 10.1 Run only after annotation

`evaluate_benchmark` returns (1) metrics with denominators and sample sizes, (2) strategy disagreement details, and (3) predictions for reproducibility. Strategy A uses accepted full identities; `PARTIAL_MATCH` is counted separately. Candidate recall requires the full canonical identity for model-only and brand+model rows, and brand identity for brand-only rows.

For B–D, end-to-end acceptance metrics are intentionally blank: candidate-generation results cannot establish a safe change in auto-match precision or coverage. All strategies use the same catalog file hash and configured matcher thresholds. A nonempty final-test split must be kept untouched while any development decisions are made.


In [ ]:
# Opt in after independently reviewing annotations in benchmark/ground_truth_template.csv.
# No reviewed real labels are currently available, so do not interpret the synthetic
# notebook datasets as evidence of real-world accuracy or strategy improvement.
if reviewed_count:
    benchmark_metrics, benchmark_disagreements, benchmark_predictions = evaluate_benchmark(GT)
    display(benchmark_metrics)
    display(benchmark_disagreements.head(50))
else:
    benchmark_metrics = pd.DataFrame()
    benchmark_disagreements = pd.DataFrame()
    benchmark_predictions = pd.DataFrame()
    print("Benchmark not run: annotation template has no reviewed records.")


### 10.2 Normalize and matching smoke check

This small check verifies the current normalization and DataFrame matching APIs; it is not evidence of accuracy.


In [ ]:
assert normalize.normalize_text("  toyota  ") == "TOYOTA"
smoke_input = pd.DataFrame({"brand": ["toyota"], "model": ["yaris"]})
smoke_result = run_pipeline(smoke_input, brand_col="brand", model_col="model")
assert smoke_result.loc[0, "corrected_brand"] == "TOYOTA"
assert smoke_result.loc[0, "corrected_model"] == "YARIS"
assert smoke_result.loc[0, "match_status"] in FULL_MATCHES
print("Normalize and matching smoke checks passed.")
